In [1]:
import mlflow

mlflow.set_tracking_uri("http://ec2-3-27-7-188.ap-southeast-2.compute.amazonaws.com:5000/")


In [2]:
mlflow.set_experiment("TFIDF-max_features")

<Experiment: artifact_location='s3://mlflow-tracking-bucket25/3', creation_time=1772892188332, experiment_id='3', last_update_time=1772892188332, lifecycle_stage='active', name='TFIDF-max_features', tags={}>

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow.sklearn
import os


In [6]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [8]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, max_features):
    # Step 2: Vectorization
    vectorizer = TfidfVectorizer(ngram_range=(3, 3), max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_type}_3-gram_{max_features}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_type}, ngram_range=(3, 3), max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", (3, 3))
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_type}, ngram_range=(3, 3), max_features={max_features}")
        plt.savefig(f"../content/confusion_matrix_{vectorizer_type}_3-gram_{max_features}.png")
        mlflow.log_artifact(f"../content/confusion_matrix_{vectorizer_type}_3-gram_{max_features}.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_type}_3-gram_{max_features}")

# Step 6: Run experiments for different max features
max_features = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]  # Example max feature size

for max_feature in max_features:

    # TF-IDF Experiments
    run_experiment("TF-IDF", max_feature)

2026/03/07 19:45:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/03/07 19:45:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run TF-IDF_3-gram_1000_RandomForest at: http://ec2-3-27-7-188.ap-southeast-2.compute.amazonaws.com:5000/#/experiments/3/runs/8caa6490c83748249c759cb040ae4a4d.
2026/03/07 19:45:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://ec2-3-27-7-188.ap-southeast-2.compute.amazonaws.com:5000/#/experiments/3.
2026/03/07 19:45:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/03/07 19:45:28 INFO mlflow.tracking._tracking_service.client: 🏃 View run TF-IDF_3-gram_2000_RandomForest at: http://ec2-3-27-7-188.ap-southeast-2.compute.amazonaws.com:5000/#/experiments/3/runs/fd583c